<a href="https://colab.research.google.com/github/Supratim0406/Text-Summarization-HuggingFace-Transformer-FastAPI/blob/main/Text_Summarization_HuggacingFace_transformer_FASTAPI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
## Mount the google drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Import Libraries

In [75]:
import pandas as pd
from transformers import T5ForConditionalGeneration, T5Tokenizer, Trainer, TrainingArguments
import warnings
warnings.filterwarnings('ignore')

## Load Dataset

In [76]:
train_data = pd.read_csv("/content/drive/MyDrive/Data/samsum-train.csv")
val_data = pd.read_csv("/content/drive/MyDrive/Data/samsum-validation.csv")

##Display a sample
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [77]:
train_data.shape, val_data.shape

((14732, 3), (818, 3))

In [78]:
## Take some sample of data for training and validation
train_data = train_data.sample(n=6000, random_state=42).reset_index(drop=True)
val_data = val_data.sample(n=500, random_state=42).reset_index(drop=True)

In [79]:
train_data.shape, val_data.shape

((6000, 3), (500, 3))

In [80]:
train_data.isnull().sum()

,0
id,0
dialogue,0
summary,0


## Data Preprocessing

In [81]:
train_data['dialogue'][0]

"Violet: hi! i came across this Austin's article and i thought that you might find it interesting\r\nViolet: <file_other>\r\nClaire: Hi! :) Thanks, but I've already read it. :)\r\nClaire: But thanks for thinking about me :)"

In [82]:
import re

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'\r\n',' ', text)  # Remove carriage and new line
    text = re.sub(r'\s+',' ', text)   # Remove extra whitespace
    text = re.sub(r'<.*?>','', text)  # Remove HTML tags
    return text.strip().lower()       # Trim and lowercase

# Clean train_data
train_data['dialogue'] = train_data['dialogue'].apply(clean_text)
train_data['summary'] = train_data['summary'].apply(clean_text)

# Clean val_data
val_data['dialogue'] = val_data['dialogue'].apply(clean_text)
val_data['summary'] = val_data['summary'].apply(clean_text)

In [83]:
## Let's see the cleaned training data
train_data.head()

,id,dialogue,summary
0,13811908,violet: hi! i came across this austin's articl...,violet sent claire austin's article.
1,13716431,pat: so does anyone know when the stream is go...,pat and lou are waiting for the stream but kev...
2,13810214,jane: jane: whaddya think? shona: this ur tin...,jane is updating her tinder profile tonight an...
3,13729823,"adam: do u have a map of paris? tom: yes, why?...",tom has a map of paris.
4,13681400,"frank: hi, how's the family? mike: great! sam'...","mike is happy, because sam's moved out. mike a..."


In [84]:
## Let's see the cleaned training data
val_data.head()

,id,dialogue,summary
0,13680857,"edd: wow, did you hear that they're transferri...",rose and edd will be transferred to a new depa...
1,13716124,"tom: where is the ""sala del capitolo"" kevin: i...","""sala del capitolo"" tom is looking for is in t..."
2,13864418,patricia: the rowing practice is cancelled! ka...,the rowing practice is cancelled. a few member...
3,13729340,"tom: u ok? alex: yeah, pretty good. u? tom: a...",tom and alex had fun last night. they drank a ...
4,13818813,"patricia: hello, here's the fair-trade brand i...",patricia recommends a fair-trade brand she tal...


## Tokenization

In [85]:
tokenizer = T5Tokenizer.from_pretrained('t5-small')

In [86]:
max_len = max(len(tokenizer.encode(text)) for text in train_data['dialogue'])
print(max_len)

Token indices sequence length is longer than the specified maximum sequence length for this model (539 > 512). Running this sequence through the model will result in indexing errors


1224


In [87]:
def preprocess_function(exmaples):
    inputs = tokenizer(exmaples['dialogue'], max_length=512, padding='max_length', truncation=True)
    targets = tokenizer(exmaples['summary'], max_length=150, padding='max_length', truncation=True)

    inputs['labels'] = targets['input_ids']
    return inputs

## Apply preprocess function on the training and and validation data
train_dataset = train_data.apply(preprocess_function, axis=1)
val_dataset = val_data.apply(preprocess_function, axis=1)

In [88]:
train_dataset[0]

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

## Fine tunning model

In [89]:
## Load model
model = T5ForConditionalGeneration.from_pretrained('t5-small')

In [90]:
# Training arguments
training_args = TrainingArguments(
    output_dir="./results",             # Output directory for checkpoints
    num_train_epochs=6,                 # no of training epochs
    per_device_train_batch_size=8,      # batch size per device during training
    per_device_eval_batch_size=8,       #  batch size per device during validation
    warmup_steps=500,                   # number of warmup steps for learning state scheduler
    weight_decay=0.01,                   # strength of weight decay
    logging_dir="./logs",               # directory for storing logs
    logging_steps=50,                   # How often to save training info
    save_steps = 500,                   # How often to save a mode checkpoint
    eval_steps = 50,                    # How often to run evaluation
    eval_strategy="epoch",              # Ensure evaluation happens after every 'epoch'
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,        # Train dataset
    eval_dataset=val_dataset            # Validation dataset
)

In [1]:
# Train
trainer.train()

NameError: name 'trainer' is not defined

In [ ]:
resutls = trainer.evaluate()
resutls

## Save and load model

In [ ]:
model.save_pretrained('/content/drive/MyDrive/Data/saved_summary_model')
tokenizer.save_pretrained('/content/drive/MyDrive/Data/saved_summary_model')

In [ ]:
#Load Model and Tokenizer
model = T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer = T5Tokenizer.from_pretrained("./saved_summary_model")

## Summarization System

In [ ]:
# Ensure the model is on the correct device (GPU if available)
device = model.device  # GPU, CPU

def summarize_dialogue(dialogue):
    dialogue = clean_text(dialogue)  # Assuming clean_text is defined

   # Tokenize and prepare inputs (pad & truncate to max length)
    inputs = tokenizer(dialogue, return_tensors="pt", truncation=True, padding="max_length", max_length=512)

    # Move tensors to device (CPU or GPU)
    inputs = {key: value.to(device) for key, value in inputs.items()}

    # Generate summary
    outputs = model.generate(
        inputs["input_ids"],
        max_length=150,  # Max length of Output
        num_beams=4,     # Provides only top 4 summary
        early_stopping=True
    )

    # Decode the generated summary
    summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return summary

In [ ]:
# Test with a dialogue on a current news topic
sample_dialogue = """
Reporter: In today's news, the latest climate change report reveals alarming global temperature rises. According to the Intergovernmental Panel on Climate Change (IPCC), the Earth’s temperature is on track to rise by 1.5°C within the next two decades.
Reporter: This is expected to lead to more frequent and severe heatwaves, flooding, and extreme weather events. Coastal cities are at particular risk due to rising sea levels.
Expert: The report emphasizes that immediate action is needed to prevent catastrophic consequences. We need to significantly reduce carbon emissions and transition to renewable energy sources.
Expert: If global temperatures increase by more than 1.5°C, we could face irreversible damage to ecosystems, agriculture, and water supply. It will have a devastating impact on biodiversity as well.
Reporter: The IPCC also stresses the importance of individual action. Governments must set stronger policies, but individuals can help by reducing waste, conserving water, and supporting green initiatives.
Expert: It's not just about the big changes; small actions like using public transportation, reducing meat consumption, and recycling can collectively make a significant difference.
Reporter: With the next UN Climate Summit coming up next month, world leaders will need to prioritize climate action. The stakes have never been higher for our planet’s future.
"""
summary = summarize_dialogue(sample_dialogue)
print("Summary:", summary)

In [ ]:
# Test with a dialogue on a different topic
sample_dialogue = """
John: Hey Sarah, have you seen the latest tech gadget reviews? I found this new smartwatch that's supposed to have amazing health tracking features.
John: It tracks heart rate, blood oxygen levels, sleep patterns, and even stress levels! It sounds like something right up your alley.
Sarah: That sounds really interesting! But I’ve been trying to cut down on tech distractions. I’ve heard these devices can be really overwhelming sometimes.
Sarah: I do think it’s cool that they can track so many health metrics though. I’m curious how accurate they really are.
John: Yeah, me too! There are also some new smartphones coming out with even better cameras and longer battery life. The new flagship model from XYZ brand has some insane specs.
Sarah: Ooh, I haven’t kept up with phones recently, but I’ve heard the camera quality is getting ridiculously good. It’s almost like a professional camera in your pocket now!
Sarah: Still, I feel like I’m fine with my current phone for now. I don’t really feel the need to upgrade unless something really groundbreaking comes out.
John: Totally understand that. It’s the same with me. But I think the battery life improvements are enough to make me consider it. I hate running out of battery when I’m out and about.
Sarah: That’s fair! I’m always worried about battery life too. Honestly, I think phones should last at least two full days on a single charge by now.
John: I agree! It’s so annoying when your phone dies in the middle of the day. I wonder if we’ll ever get to a point where we don’t have to charge our phones every day.
Sarah: That would be amazing! I think as tech improves, battery tech might also catch up. Let’s hope the next generation of phones can last longer!
"""

summary = summarize_dialogue(sample_dialogue)
print("Summary:", summary)

In [ ]:
# Test with a sample input
sample_dialogue = """
Violet: Hey Claire! I was reading an article about Austin and thought you might find it interesting!
Violet: It's about the current trends in urban development and how cities are planning for the future.
Violet: Here, let me share the link: <file_other>
Claire: Oh wow, that sounds like an insightful read. But I've actually already read that one last week.
Claire: It was really interesting though, especially the part about sustainable architecture in cities.
Claire: You know, I've been following these urban planning discussions for a while now.
Violet: Oh, I didn’t know that! Well, I’ll look for something else then, maybe something about eco-friendly cities or tech innovations.
Claire: That would be awesome! Let me know if you find something cool.
Violet: Sure, I’ll keep you posted. Thanks for the feedback!
"""

summary = summarize_dialogue(sample_dialogue)
print("Summary:", summary)

## Download Model to you Machine

In [ ]:
import shutil

# Path to the directory containing the fine-tuned model
model_dir = "/content/drive/MyDrive/Data/"

# Output zip file path
output_zip_path = "saved_summary_model.zip"

# Create a zip archive
shutil.make_archive(base_name="saved_summary_model", format="zip", root_dir=model_dir)